# Population density, 10000 BCE – 2100

Our World in Data country-level **people per km²**, not HYDE rasters. The download is a CSV (`population-density.csv`), same columns as the usual OWID Excel export.

- **10000 BCE–1799:** HYDE v3.3  
- **1800–1949:** Gapminder v7  
- **1950–2023:** UN WPP 2024  
- **2024–2100:** UN WPP medium-variant *projections*

Density = population ÷ FAO land area. Source notes: [readme copy.md](readme%20copy.md).

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

df = pd.read_csv("population-density.csv")
print(df.columns.tolist())
print(df.shape)
print("Year range:", int(df["Year"].min()), "to", int(df["Year"].max()))
print("Entities:", df["Entity"].nunique())
df.head()

['Entity', 'Code', 'Year', 'Population density']
(76576, 4)
Year range: -10000 to 2100
Entities: 251


,Entity,Code,Year,Population density
0,Afghanistan,AFG,-10000,0.022595
1,Afghanistan,AFG,-9000,0.031285
2,Afghanistan,AFG,-8000,0.043318
3,Afghanistan,AFG,-7000,0.059979
4,Afghanistan,AFG,-6000,0.083047


Plotly choropleths need ISO-3 country codes (`AFG`, `IND`, …). Rows with `OWID_*` codes are continents, income groups, and former states — drop them.

There are 341 years in the file (sparse before 1950, then annual through 2100). Using every annual frame makes the animation huge, so after 1950 we keep **every 10th year plus the last year**. Color is **log10(density)** with a **fixed** scale so later centuries look denser instead of the colorbar resetting each frame.

In [2]:
iso = df[df["Code"].str.fullmatch(r"[A-Z]{3}", na=False)].copy()
year_min = int(iso["Year"].min())
year_max = int(iso["Year"].max())
years = np.array(sorted(iso["Year"].unique()))
keep_years = [y for y in years if y < 1950 or y % 10 == 0 or y == year_max]

df_map = iso[iso["Year"].isin(keep_years)].copy()
df_map["density_log"] = np.log10(df_map["Population density"].clip(lower=1e-6))

print(f"Countries: {df_map['Code'].nunique()}")
print(f"Frames: {len(keep_years)}  ({year_min} → {year_max})")
print("First / last frames:", keep_years[:8], "…", keep_years[-5:])

Countries: 234
Frames: 206  (-10000 → 2100)
First / last frames: [np.int64(-10000), np.int64(-9000), np.int64(-8000), np.int64(-7000), np.int64(-6000), np.int64(-5000), np.int64(-4000), np.int64(-3000)] … [np.int64(2060), np.int64(2070), np.int64(2080), np.int64(2090), np.int64(2100)]


In [3]:
fig = px.choropleth(
    df_map,
    locations="Code",
    color="density_log",
    hover_name="Entity",
    hover_data={
        "Population density": ":.3f",
        "Year": True,
        "Code": False,
        "density_log": False,
    },
    animation_frame="Year",
    color_continuous_scale="YlOrRd",
    range_color=(df_map["density_log"].quantile(0.02), df_map["density_log"].quantile(0.98)),
    title=f"Population density by country ({abs(year_min)} BCE – {year_max} CE)",
    labels={"density_log": "people / km² (log10)", "Year": "Year"},
)

fig.update_layout(margin={"r": 0, "t": 50, "l": 0, "b": 0})
fig.update_geos(showframe=False, showcoastlines=True)
fig.show()